# Sistema LADA v4 - Deployment

**Learning Analytics and Decision Advisor**

Sistema de recomendación académica basado en clustering multinivel para predecir probabilidades de éxito estudiantil.

---

## Contenido
1. Carga del modelo entrenado
2. Definición de funciones auxiliares
3. Ejemplo práctico de uso
4. Guía de implementación

## Setup Inicial

Importamos las librerías necesarias para el funcionamiento del sistema.

In [ ]:
import pickle
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Sistema LADA v4 - Deployment")
print("-" * 50)

## 1. Carga del Modelo

Cargamos el modelo previamente entrenado junto con todos sus componentes necesarios.

In [ ]:
print("Cargando modelo entrenado...")

with open('../models_2/lada_modelo_v4.pkl', 'rb') as archivo:
    modelo_lada = pickle.load(archivo)

resultados_por_nivel = modelo_lada['resultados_por_nivel']
df_estudiantes = modelo_lada['df_estudiantes']
df_train = modelo_lada['df_train']
df_facultades_departamentos = modelo_lada['df_facultades_departamentos']
usar_departamento_nivel2 = modelo_lada['usar_departamento_nivel2']
metadata = modelo_lada['metadata']

print("\nModelo cargado correctamente")
print(f"Versión: {metadata['version']}")
print(f"Fecha de entrenamiento: {metadata['fecha_entrenamiento']}")
print(f"\nMétricas de desempeño:")
print(f"  - Error Absoluto Medio (MAE): {metadata['metricas']['mae']*100:.2f}%")
print(f"  - Accuracy (±10%): {metadata['metricas']['accuracy_10']*100:.1f}%")
print(f"  - Accuracy (±20%): {metadata['metricas']['accuracy_20']*100:.1f}%")

## 2. Funciones del Sistema

Definimos las funciones necesarias para hacer predicciones con el modelo.

In [ ]:
def extraer_categoria_curso(codigo_curso, df_facultades_departamentos, usar_departamento=False):
    """
    Extrae la categoría de un curso basándose en su facultad o departamento.
    
    Args:
        codigo_curso: Código del curso (ej: 'CRS_00001234')
        df_facultades_departamentos: DataFrame con mapeo curso-facultad-departamento
        usar_departamento: Si True usa departamento, si False usa facultad
    
    Returns:
        String con la categoría (ej: 'FAC_001' o 'DEPT_005')
    """
    if pd.isna(codigo_curso):
        return 'DESCONOCIDO'
    
    match = df_facultades_departamentos[
        df_facultades_departamentos['CODIGO_CURSO'] == codigo_curso
    ]
    
    if len(match) == 0:
        return 'DESCONOCIDO'
    
    if usar_departamento:
        codigo = match['CODIGO_DEPARTAMENTO'].iloc[0]
        return f"DEPT_{codigo}" if pd.notna(codigo) else 'DESCONOCIDO'
    else:
        codigo = match['CODIGO_FACULTAD'].iloc[0]
        return f"FAC_{codigo}" if pd.notna(codigo) else 'DESCONOCIDO'

In [ ]:
def seleccionar_nivel_adaptativo(estudiante_perfil, df_inscripciones, 
                                 resultados_por_nivel, df_facultades_departamentos, 
                                 usar_departamento=False):
    """
    Selecciona el nivel jerárquico más apropiado para hacer la predicción.
    
    El sistema intenta usar el nivel más específico posible (3 → 2 → 1):
    - NIVEL_3: Combinación exacta de cursos (más específico)
    - NIVEL_2: Distribución por categorías de cursos
    - NIVEL_1: Número total de cursos (menos específico)
    
    Args:
        estudiante_perfil: Diccionario con información del estudiante
        df_inscripciones: DataFrame con inscripciones históricas
        resultados_por_nivel: Diccionario con clusters por nivel
        df_facultades_departamentos: DataFrame con mapeo curso-facultad
        usar_departamento: Usar departamento en lugar de facultad
    
    Returns:
        Tupla (nivel, razón, info_cluster)
    """
    cursos = estudiante_perfil.get('cursos', [])
    num_cursos = len(cursos)
    
    firma_nivel_1 = str(num_cursos)
    
    categorias = [extraer_categoria_curso(c, df_facultades_departamentos, usar_departamento) 
                  for c in cursos]
    contador = pd.Series(categorias).value_counts().to_dict()
    firma_nivel_2 = '_'.join([f"{cat}:{count}" for cat, count in sorted(contador.items())])
    
    cursos_ordenados = sorted([str(c) for c in cursos])
    firma_nivel_3 = '_'.join(cursos_ordenados)
    
    niveles = [
        ('NIVEL_3', firma_nivel_3, 20),
        ('NIVEL_2', firma_nivel_2, 10),
        ('NIVEL_1', firma_nivel_1, 5)
    ]
    
    for nivel_nombre, firma, min_casos in niveles:
        clusters = resultados_por_nivel.get(nivel_nombre, {}).get('clusters', {})
        
        if firma in clusters:
            info_cluster = clusters[firma]
            total_casos = info_cluster['total_casos']
            
            if total_casos >= min_casos:
                razon = f"Se encontraron {total_casos} casos similares en {nivel_nombre}"
                return nivel_nombre, razon, info_cluster
    
    razon = "Caso atípico: no hay suficientes datos históricos"
    clusters_nivel1 = resultados_por_nivel.get('NIVEL_1', {}).get('clusters', {})
    info_cluster = clusters_nivel1.get(firma_nivel_1, None)
    
    return 'NIVEL_1', razon, info_cluster

In [ ]:
def predecir_probabilidad_exito(estudiante_perfil, df_inscripciones, 
                               resultados_por_nivel, df_facultades_departamentos, 
                               usar_departamento=False):
    """
    Predice la probabilidad de que un estudiante apruebe todos sus cursos.
    
    Args:
        estudiante_perfil: Diccionario con:
            - cursos: Lista de códigos de curso
            - num_cursos: Número de cursos
            - creditos: Total de créditos
            - pga_anterior: PGA del periodo anterior
            - semestres_anteriores: Semestres cursados
            - pct_creditos_anterior: % de créditos aprobados
        df_inscripciones: DataFrame con datos históricos
        resultados_por_nivel: Diccionario con clusters
        df_facultades_departamentos: DataFrame con mapeo
        usar_departamento: Usar departamento en lugar de facultad
    
    Returns:
        Diccionario con la predicción y metadata
    """
    nivel, razon, info_cluster = seleccionar_nivel_adaptativo(
        estudiante_perfil, df_inscripciones, resultados_por_nivel,
        df_facultades_departamentos, usar_departamento
    )
    
    if info_cluster is None:
        return {
            "nivel_usado": nivel,
            "razon": razon,
            "probabilidad_exito": None,
            "cluster_id": None,
            "num_estudiantes_similares": 0,
            "confianza": None,
            "mensaje": "No hay datos históricos suficientes para esta combinación de cursos"
        }
    
    pga_anterior = estudiante_perfil.get("pga_anterior", None)
    semestres_anteriores = estudiante_perfil.get("semestres_anteriores", None)
    pct_creditos_anterior = estudiante_perfil.get("pct_creditos_anterior", None)
    num_cursos = estudiante_perfil.get("num_cursos", None)
    creditos = estudiante_perfil.get("creditos", None)
    
    if pga_anterior is None or pd.isna(pga_anterior):
        return {
            "nivel_usado": nivel,
            "razon": razon,
            "probabilidad_exito": None,
            "mensaje": "No hay información de PGA anterior para este estudiante"
        }
    
    features = np.array([
        pga_anterior,
        semestres_anteriores if semestres_anteriores is not None else 0,
        pct_creditos_anterior if pct_creditos_anterior is not None else 100.0,
        num_cursos,
        creditos
    ])
    
    scaler = info_cluster.get("scaler")
    if scaler is not None:
        features_normalizadas = scaler.transform([features])[0]
    else:
        features_normalizadas = features
    
    centroides = info_cluster["centroides"]
    distancias = np.linalg.norm(centroides - features_normalizadas, axis=1)
    
    cluster_id = int(np.argmin(distancias))
    
    probabilidad = info_cluster["tasas_exito"][cluster_id]
    num_similares = info_cluster["tamanos_cluster"][cluster_id]
    
    if num_similares >= 50:
        confianza = "ALTA"
    elif num_similares >= 20:
        confianza = "MEDIA"
    else:
        confianza = "BAJA"
    
    return {
        "nivel_usado": nivel,
        "razon": razon,
        "probabilidad_exito": probabilidad,
        "cluster_id": cluster_id,
        "num_estudiantes_similares": num_similares,
        "confianza": confianza,
        "estudiantes_similares": info_cluster["estudiantes_por_cluster"][cluster_id],
        "algoritmo": info_cluster.get("algoritmo", "fuzzy")
    }

print("Funciones del sistema cargadas correctamente")

## 3. Ejemplo de Uso

Demostramos cómo usar el sistema con un caso de ejemplo.

In [ ]:

estudiante_ejemplo = {
    'estudiante_id': 'EST_EJEMPLO_001',
    'cursos': ['CRS_00001234', 'CRS_00005678', 'CRS_00009012'],
    'num_cursos': 3,
    'creditos': 12,
    'pga_anterior': 3.5,
    'semestres_anteriores': 2,
    'pct_creditos_anterior': 85.0,
    'fuente_pga': 'PGA_HISTORICO'
}

print("Realizando predicción para estudiante de ejemplo...\n")

resultado = predecir_probabilidad_exito(
    estudiante_ejemplo,
    df_train,
    resultados_por_nivel,
    df_facultades_departamentos,
    usar_departamento_nivel2
)

print("Resultados de la predicción:")
print("-" * 50)
print(f"Nivel jerárquico utilizado: {resultado['nivel_usado']}")
print(f"Algoritmo de clustering: {resultado.get('algoritmo', 'N/A').upper()}")

if resultado['probabilidad_exito'] is not None:
    print(f"\nProbabilidad de éxito: {resultado['probabilidad_exito']*100:.1f}%")
    print(f"Nivel de confianza: {resultado['confianza']}")
    print(f"Estudiantes con perfil similar: {resultado['num_estudiantes_similares']}")
    print(f"\nExplicación: {resultado['razon']}")
else:
    print(f"\nNo se pudo generar predicción: {resultado.get('mensaje', 'Error desconocido')}")

## 4. Guía de Implementación

### Formato de entrada

Para hacer una predicción, necesitas crear un diccionario con la siguiente estructura:

```python
perfil_estudiante = {
    'estudiante_id': 'codigo_estudiante',     # Identificador único
    'cursos': ['CRS_001', 'CRS_002'],        # Lista de códigos de curso
    'num_cursos': 2,                         # Número de cursos
    'creditos': 8,                           # Total de créditos
    'pga_anterior': 3.8,                     # PGA del periodo anterior (0-5)
    'semestres_anteriores': 3,               # Semestres cursados previamente
    'pct_creditos_anterior': 90.0            # % de créditos aprobados (0-100)
}
```

### Hacer una predicción

```python
resultado = predecir_probabilidad_exito(
    perfil_estudiante,
    df_train,
    resultados_por_nivel,
    df_facultades_departamentos,
    usar_departamento_nivel2
)
```

### Interpretar resultados

- **probabilidad_exito**: Valor entre 0 y 1 (multiplicar por 100 para porcentaje)
- **confianza**: ALTA (≥50 casos), MEDIA (≥20 casos), BAJA (<20 casos)
- **nivel_usado**: NIVEL_3 (más específico) → NIVEL_2 → NIVEL_1 (menos específico)
- **algoritmo**: fuzzy, kmeans o gmm (el que mejor funcionó para esa combinación)

In [ ]:
print("\nSistema LADA v4 listo para uso en producción")
print("-" * 50)
print("Usa la función predecir_probabilidad_exito() para generar nuevas predicciones")